In [1]:
from IPython.display import display 

import base64
import json
import requests
from datetime import datetime, timedelta
from tqdm import tqdm

import pandas as pd 
pd.set_option('display.float_format', '{:.00f}'.format)

import os 
import numpy as np
import ast
from sympy import symbols, solve, lambdify

import missingno as msno
import matplotlib.pyplot as plt


In [2]:
import sys
from pathlib import Path

try:
    # Works in Python scripts
    helper_path = Path(__file__).resolve().parent.parent / "helper"
except NameError:
    # Works in Jupyter notebooks
    helper_path = Path().resolve().parent / "helper"

sys.path.insert(0, str(helper_path))

# Now import your modules
from config import gam_info

from security_config import emplifi_key
from functions import execute_sql_query
import test_functions

In [3]:
platformID = 'TTK'

# country
country_cols = ['PlaceID',	'TikTok Codes', gam_info['population_column']]
country_codes = pd.read_excel(f"../../{gam_info['lookup_file']}", 
                              sheet_name='CountryID', usecols=country_cols, keep_default_na=False )

# week 
week_tester = pd.read_excel(f"../../{gam_info['lookup_file']}", 
                            sheet_name='GAM Period', keep_default_na=False)
week_tester['w/c'] = pd.to_datetime(week_tester['w/c'])

# social media accounts
dtype_dict = {'Channel ID': 'str',
              'Linked FB Account': 'str'}
socialmedia_accounts = pd.read_excel(f"../../{gam_info['lookup_file']}", dtype=dtype_dict,
                                     sheet_name='Social Media Accounts new', keep_default_na=False)

socialmedia_accounts = socialmedia_accounts[(socialmedia_accounts['PlatformID'] == platformID)
                                            & 
                                            (socialmedia_accounts['Status'] == 'active')]
socialmedia_accounts = socialmedia_accounts.rename(columns={'Excluding UK': 'Channel Group'})

channel_ids = socialmedia_accounts['Channel ID'].unique().tolist()
formatted_channel_ids = ', '.join(f"'{channel_id}'" for channel_id in channel_ids)

# service
cols = ['ServiceID', 'Lumen']
service_codes = pd.read_excel(f"../../{gam_info['lookup_file']}", 
                              sheet_name='ServiceID', usecols=cols, keep_default_na=False )

### RUN TESTS
test_functions.test_lookup_files(country_codes, ['PlaceID'], [0,1,2], 
                                 test_step="lookup files - ensuring country codes is correct")

test_functions.test_lookup_files(week_tester, ['w/c'], [3,4,5], 
                                 test_step = "lookup files - ensuring week tester is correct")

test_functions.test_lookup_files(socialmedia_accounts, ['Channel ID'], [6,7,8], 
                                 test_step = "lookup files - ensuring social media accounts is correct")


test_functions.test_lookup_files(service_codes, ['ServiceID', 'Lumen'], [9,10,11], 
                                 test_step = "lookup files - ensuring service is correct")



✅ Test 0 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test 1 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test 2 passed: No missing values in lookup.
...updating logbook...

✅ Test 3 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test 4 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test 5 passed: No missing values in lookup.
...updating logbook...

✅ Test 6 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test 7 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test 8 passed: No missing values in lookup.
...updating logbook...

✅ Test 9 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test 10 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test 11 passed: No missing values in lookup.
...updating logbook...



# read in 

In [4]:

dataframes = []
storage_dir = f"../data/raw/{platformID}/post_level/"
csv_files = [f for f in os.listdir(storage_dir) if f.endswith(".csv")]
for f in csv_files:
    file_path = os.path.join(storage_dir, f)
    try:
        parts = f.replace(".csv", "").split("_")
        file_timeinfo = parts[0]
        platformID = parts[1]
        profile_id = parts[2]
        week_str = parts[3]

        df = pd.read_csv(file_path)
        df["platformID"] = platformID
        df["profile_id"] = profile_id
        df["w/c"] = week_str
        
        if not df.empty:
            dataframes.append(df)
    except pd.errors.EmptyDataError:
        print(f"❌ Could not read file (empty or malformed): {f}")

# Combine all non-empty DataFrames
if dataframes:
    post_level_df = pd.concat(dataframes, ignore_index=True)
    print("✅ Combined DataFrame created.")
    display(post_level_df.head())
else:
    print("🚫 No valid data found to combine.")


✅ Combined DataFrame created.


,attachments,author,authorId,content_type,created_time,duration,id,link,media,message,...,insights_likes,insights_reach,insights_reach_engagement_rate,insights_shares,insights_video_views,insights_view_time,insights_viewers_by_country,platformID,profile_id,w/c
0,[{'image_url': 'https://p16-pu-sign-no.tiktokc...,"{'id': '3d596e6a-d239-434e-bb79-93e5d291b216',...",3d596e6a-d239-434e-bb79-93e5d291b216,post,2025-08-10T19:00:00+00:00,51,7536981698806811926,https://www.tiktok.com/@bbcnews/video/75369816...,video,Japan’s prime minister has described the count...,...,47374,499763,10,662,603159,7895667,"[{'country': 'CA', 'percentage': 0.039}, {'cou...",TTK,3d596e6a-d239-434e-bb79-93e5d291b216,2025-08-04
1,[{'image_url': 'https://p16-pu-sign-no.tiktokc...,"{'id': '3d596e6a-d239-434e-bb79-93e5d291b216',...",3d596e6a-d239-434e-bb79-93e5d291b216,post,2025-08-10T17:00:00+00:00,14,7536980939285499158,https://www.tiktok.com/@bbcnews/video/75369809...,video,A meteorite that crashed into a home in the US...,...,1043847,9793095,11,24274,11863167,115011336,"[{'country': 'GB', 'percentage': 0.126}, {'cou...",TTK,3d596e6a-d239-434e-bb79-93e5d291b216,2025-08-04
2,[{'image_url': 'https://p16-pu-sign-no.tiktokc...,"{'id': '3d596e6a-d239-434e-bb79-93e5d291b216',...",3d596e6a-d239-434e-bb79-93e5d291b216,post,2025-08-10T15:15:37+00:00,251,7536975832674372886,https://www.tiktok.com/@bbcnews/video/75369758...,video,Are planes actually crashing more often? #Plan...,...,51630,509953,11,1853,594159,9300057,"[{'country': 'GH', 'percentage': 0.035}, {'cou...",TTK,3d596e6a-d239-434e-bb79-93e5d291b216,2025-08-04
3,[{'image_url': 'https://p16-pu-sign-no.tiktokc...,"{'id': '3d596e6a-d239-434e-bb79-93e5d291b216',...",3d596e6a-d239-434e-bb79-93e5d291b216,post,2025-08-10T14:32:50+00:00,18,7536964795443039510,https://www.tiktok.com/@bbcnews/video/75369647...,video,Israeli Prime Minister Benjamin Netanyahu says...,...,83953,2519274,4,4360,3229541,45168020,"[{'country': 'NG', 'percentage': 0.031}, {'cou...",TTK,3d596e6a-d239-434e-bb79-93e5d291b216,2025-08-04
4,[{'image_url': 'https://p16-pu-sign-no.tiktokc...,"{'id': '3d596e6a-d239-434e-bb79-93e5d291b216',...",3d596e6a-d239-434e-bb79-93e5d291b216,post,2025-08-10T13:00:00+00:00,67,7536900507957349654,https://www.tiktok.com/@bbcnews/video/75369005...,video,"Zara, Next and Marks & Spencer have all had ad...",...,70891,589456,12,1725,781333,9648022,"[{'country': 'US', 'percentage': 0.121}, {'cou...",TTK,3d596e6a-d239-434e-bb79-93e5d291b216,2025-08-04


In [5]:
post_level_df = post_level_df.rename(columns={'platformID': 'PlatformID'})
# Count unique weeks per profile
week_counts = post_level_df.groupby('profile_id')['w/c'].nunique().reset_index()
week_counts.columns = ['profile_id', 'number_of_weeks']

# Optional: sort by number of weeks
week_counts = week_counts.sort_values(by='number_of_weeks', ascending=False)

# Display 
print(week_counts)

# Assuming post_level_df is already loaded and contains 'profile_id' and 'w/c'
# Create a pivot table: profiles as rows, weeks as columns
pivot_df = post_level_df.pivot_table(columns='profile_id', index='w/c', aggfunc='size')

# Convert to boolean: True = data exists, False = missing
#pivot_df = pivot_df.astype(bool)

# Visualize missing data
#msno.matrix(pivot_df)
#plt.title("Missing Weeks per TikTok Profile")
#plt.show()

                              profile_id  number_of_weeks
0   057b1686-d9f3-51fe-a129-5732678e609e               34
2   2ac83179-5aa1-4b36-9a7c-a8418f72a799               34
4   34e41ced-576d-430a-b590-55d0c1d241b1               34
7   4be46cb0-a590-5e23-b3cf-ea59dd02c530               34
6   3d596e6a-d239-434e-bb79-93e5d291b216               34
8   54ac56ff-37ee-597e-89ec-d63de04c9df1               34
14  c02ca653-c3b6-4b34-b210-711e12f9eb2d               34
12  a53907d1-2b3d-57a1-be38-cea14a04dc0f               34
9   746ce2b5-197a-4eb6-86d2-aa2a39a021f7               34
15  c3390a5e-42f2-4652-99ff-b903b61d8fc7               34
13  a824bd24-fa59-4039-88bf-4b152ffa2881               34
5   3c4d1098-0742-41ed-9182-f7ea05f398cd               22
3   2bf05522-b1ed-5751-a294-22f1d95f6cd3               18
10  80244afe-5625-509b-9839-0c5dfbc95d95               16
11  a1b31ad8-b2c1-5123-a508-d01883306385               14
16  fbd019f7-9dcb-5f22-a0e0-86cfb49a5959               13
1   1ea28c7a-b

In [6]:

def extract_author_info(row):
    if pd.isna(row):
        return pd.Series({'id': None, 'name': None, 'url': None})

    if isinstance(row, str):
        try:
            author_dict = ast.literal_eval(row)
        except (ValueError, SyntaxError):
            return pd.Series({'id': None, 'name': None, 'url': None})
    elif isinstance(row, dict):
        author_dict = row
    else:
        return pd.Series({'id': None, 'name': None, 'url': None})

    return pd.Series({
        'id': author_dict.get('id'),
        'name': author_dict.get('name'),
        'url': author_dict.get('url')
    })

# Apply the function
post_level_df[["Channel ID", "Channel Name", "Channel URL"]] = post_level_df['author'].apply(extract_author_info)

# Views

In [7]:

minnie_cols_used = {'Date': 'w/c', #minnie has a day by day breakdown and then calculates the average
               'Profile ID': "Channel ID", # author['name'],
               'Profile name': "Channel Name", # author['url'],
               'Profile URL': "Channel URL", #author['id'],
               #'Post detail URL': 'link',
               #'Content ID': 'link', # splice out from link
               'Platform': 'PlatformID', 
               'Content type': 'content_type',
               'Media type': 'media',
               #'Title': '', # missing
               #'Description': '', # missing
               'Content': 'message',
               #'Link URL': '', #unclear
               'View on platform': 'link',
               'Engagements': 'insights_engagements',
               'Total reach': 'insights_reach', #but number is different? 
               'Video length (sec)': 'duration',
               'Video view count': 'insights_video_views',
               'Total video view time (sec)': 'insights_view_time',
               'Average time watched (sec)': 'insights_avg_time_watched',
               'Completion rate': 'insights_completion_rate',
              }

views_df = post_level_df[minnie_cols_used.values()]
views_df['link'] = views_df['link'].fillna('').astype(str)
views_df['content_id'] = views_df['link'].str.split('/').str[-1].str.split('?').str[0]
views_df.head()

/var/folders/gz/pq5c3fbj5rs1tz_5w1hycq4h0000gn/T/ipykernel_70662/3751647070.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  views_df['link'] = views_df['link'].fillna('').astype(str)
/var/folders/gz/pq5c3fbj5rs1tz_5w1hycq4h0000gn/T/ipykernel_70662/3751647070.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  views_df['content_id'] = views_df['link'].str.split('/').str[-1].str.split('?').str[0]


,w/c,Channel ID,Channel Name,Channel URL,PlatformID,content_type,media,message,link,insights_engagements,insights_reach,duration,insights_video_views,insights_view_time,insights_avg_time_watched,insights_completion_rate,content_id
0,2025-08-04,3d596e6a-d239-434e-bb79-93e5d291b216,BBC News,https://www.tiktok.com/@bbcnews,TTK,post,video,Japan’s prime minister has described the count...,https://www.tiktok.com/@bbcnews/video/75369816...,49497,499763,51,603159,7895667,13,0,7536981698806811926
1,2025-08-04,3d596e6a-d239-434e-bb79-93e5d291b216,BBC News,https://www.tiktok.com/@bbcnews,TTK,post,video,A meteorite that crashed into a home in the US...,https://www.tiktok.com/@bbcnews/video/75369809...,1076336,9793095,14,11863167,115011336,10,0,7536980939285499158
2,2025-08-04,3d596e6a-d239-434e-bb79-93e5d291b216,BBC News,https://www.tiktok.com/@bbcnews,TTK,post,video,Are planes actually crashing more often? #Plan...,https://www.tiktok.com/@bbcnews/video/75369758...,54166,509953,251,594159,9300057,16,0,7536975832674372886
3,2025-08-04,3d596e6a-d239-434e-bb79-93e5d291b216,BBC News,https://www.tiktok.com/@bbcnews,TTK,post,video,Israeli Prime Minister Benjamin Netanyahu says...,https://www.tiktok.com/@bbcnews/video/75369647...,102324,2519274,18,3229541,45168020,14,0,7536964795443039510
4,2025-08-04,3d596e6a-d239-434e-bb79-93e5d291b216,BBC News,https://www.tiktok.com/@bbcnews,TTK,post,video,"Zara, Next and Marks & Spencer have all had ad...",https://www.tiktok.com/@bbcnews/video/75369005...,73598,589456,67,781333,9648022,13,0,7536900507957349654


In [8]:


# optional: test video length is all in seconds
print(f"number of entries: {views_df.shape}")
views_df = views_df[~views_df['insights_reach'].isna()]
print(f"number of entries that have reach: {views_df.shape}")

cols_fill_nan = ['insights_avg_time_watched', 'duration', 'insights_reach',
                 'insights_completion_rate']
views_df[cols_fill_nan] = views_df[cols_fill_nan].fillna(0)  # or any other value you'd like


number of entries: (12084, 17)
number of entries that have reach: (12084, 17)


In [9]:
# Define x and y values for each row
views_df['x1'] = 0
views_df['x2'] = views_df['insights_avg_time_watched']
views_df['x3'] = views_df['duration']
views_df['x output'] = 10

views_df['y1'] = views_df['insights_reach']
views_df['y2'] = views_df['insights_reach'] / 2
views_df['y3'] = views_df['insights_reach'] * views_df['insights_completion_rate']

x_sym, a_sym, b_sym, c_sym = symbols('x a b c')

def find_quadratic_coefficients(point1, point2, point3):
    x1, y1 = point1
    x2, y2 = point2
    x3, y3 = point3
    
    if len({x1, x2, x3}) < 3:
        return None

    y_expr = a_sym * x_sym**2 + b_sym * x_sym + c_sym

    eq1 = y_expr.subs(x_sym, x1) - y1
    eq2 = y_expr.subs(x_sym, x2) - y2
    eq3 = y_expr.subs(x_sym, x3) - y3

    solutions = solve((eq1, eq2, eq3), (a_sym, b_sym, c_sym))
    return solutions


def apply_quadratic(row):
    if row['x3'] == 0:
        return 100
    if len(set([row['x1'], row['x2'], row['x3']])) < 3:
    # Handle the case where any two x-values are the same
        return 100

    point1 = (row['x1'], row['y1'])
    point2 = (row['x2'], row['y2'])
    point3 = (row['x3'], row['y3'])
    x_val = row['x output']

    coeffs = find_quadratic_coefficients(point1, point2, point3)
    if coeffs is None:
        return np.nan

    # If all y-values are zero, use only the b coefficient
    if row['y1'] == 0 and row['y2'] == 0 and row['y3'] == 0:
        return 10 * coeffs[b_sym]

    # Otherwise, evaluate full quadratic
    y_expr = a_sym * x_sym**2 + b_sym * x_sym + c_sym
    y_val = y_expr.subs({a_sym: coeffs[a_sym], b_sym: coeffs[b_sym], c_sym: coeffs[c_sym], x_sym: x_val})
    return y_val


# Create new column with interpolated values
views_df['30sec_video_views'] = views_df.apply(apply_quadratic, axis=1).astype(float)
views_df['completed_video_views'] = views_df['insights_completion_rate'] * views_df['insights_reach']

conditions = [
    views_df['insights_reach'] == 0,
    (views_df['completed_video_views'].round(0) > views_df['30sec_video_views'].round(0)),
    views_df['insights_avg_time_watched'] <= 10,
    (views_df['insights_avg_time_watched'] > 10) & (views_df['insights_avg_time_watched'] <= 15),
    views_df['insights_reach'] < views_df['30sec_video_views'],
    views_df['duration'] == 0
]

choices = [
    views_df['insights_engagements'],
    views_df['completed_video_views'],
    views_df['insights_engagements'],
    views_df['insights_reach'] * 0.67,
    views_df['insights_reach'] * 0.799,
    views_df['insights_engagements']
]

views_df['final_video_views'] = np.select(conditions, choices, 
                                            default=views_df['30sec_video_views'])

In [13]:
file_path = f"../data/processed/{platformID}"
os.makedirs(file_path, exist_ok=True)

cols = ['content_id', 'ServiceID', 'Channel ID', 'Channel Name', 'w/c', 
        'link',
        'final_video_views', 'Linked FB Account'
       ]
views_df = views_df[cols]
views_df.to_csv(f"{file_path}/{gam_info['file_timeinfo']}_{platformID}_views.csv",
                       index=None)


In [14]:
# YT views per viewer is missing for media action 
yt_views_per_viewer = pd.read_excel("../helper/YT views per viewer_TTKhelper.xlsx")[['w/c', 'Service', 'Value']]
yt_views_per_viewer = yt_views_per_viewer.rename(columns={'Service': 'Lumen', 'Value': 'views_per_viewer'})
yt_views_per_viewer = yt_views_per_viewer.merge(service_codes, on='Lumen', how='left').drop(columns='Lumen')
yt_views_per_viewer.sample()

,w/c,views_per_viewer,ServiceID
1229,2024-04-22,2,POR


In [15]:
views_df['w/c'] = pd.to_datetime(views_df['w/c'])
yt_views_per_viewer['w/c'] = pd.to_datetime(yt_views_per_viewer['w/c'])

views_df_yt = views_df.merge(yt_views_per_viewer, on=['ServiceID', 'w/c'], how='left', indicator=True)

matched = views_df_yt[views_df_yt['_merge'] == 'both'].drop(columns='_merge')
unmatched = views_df_yt[views_df_yt['_merge'] == 'left_only'].drop(columns=['_merge', 'views_per_viewer'])

views_per_viewer_by_service = yt_views_per_viewer.groupby(['ServiceID'])['views_per_viewer'].mean().reset_index()

matched_sec = unmatched.merge(views_per_viewer_by_service, on='ServiceID', how='left', indicator=True)
matched_sec = matched_sec[matched_sec['_merge'] == 'both'].drop(columns='_merge')

views_scaled = pd.concat([matched, matched_sec])
views_scaled['engaged_users'] = views_scaled['final_video_views']/(views_scaled['views_per_viewer']*1.14)
views_scaled.columns

Index(['content_id', 'ServiceID', 'Channel ID', 'Channel Name', 'w/c', 'link',
       'final_video_views', 'Linked FB Account', 'views_per_viewer',
       'engaged_users'],
      dtype='object')

# Country 

In [16]:
country_df = post_level_df.copy()

In [17]:

# Step 1: Parse the stringified list of country-percentage dictionaries
country_df['parsed_viewers'] = country_df['insights_viewers_by_country'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else []
)

# Step 2: Explode the parsed list into separate rows
exploded_df = country_df.explode('parsed_viewers').reset_index(drop=True)

# Step 3: Extract 'country' and 'percentage' from each dictionary
exploded_df[['country', 'percentage']] = exploded_df['parsed_viewers'].apply(
    lambda entry: pd.Series({
        'country': entry.get('country') if isinstance(entry, dict) else None,
        'percentage': entry.get('percentage') if isinstance(entry, dict) else None
    })
)

# Step 4: Drop the intermediate column
exploded_df.drop(columns=['parsed_viewers'], inplace=True)
exploded_df['country'] = exploded_df['country'].replace('Others', 'ZZ')
exploded_df['country'] = exploded_df['country'].fillna('ZZ')


In [18]:

exploded_df = exploded_df.rename(columns={'country': 'TikTok Codes'})
ttk_country_all = exploded_df.merge(country_codes[['TikTok Codes', 'PlaceID']], on='TikTok Codes', how='left',
                 indicator=True)

print(f"mismatches? \n{ttk_country_all._merge.value_counts()}")
ttk_country_all = ttk_country_all.drop(columns='_merge')

# Remove unknown countries (UNK)
ttk_country = ttk_country_all[ttk_country_all['PlaceID'] != 'UNK'].copy()

# Compute sum of percentages per video
sum_per_video = ttk_country.groupby('id')['percentage'].transform('sum')

# Rescale percentages
ttk_country['rescaled_percentage'] = ttk_country['percentage'] / sum_per_video

# Optional: Check that each video sums to 1
check = ttk_country.groupby('id')['rescaled_percentage'].sum()
check[~np.isclose(check, 1, atol=1e-6)]

country_cols = ['Channel ID', 'link', 'PlaceID', 'rescaled_percentage', 'w/c', 'PlatformID']
ttk_country= ttk_country[country_cols]
ttk_country.to_csv(f"../data/processed/{platformID}/{gam_info['file_timeinfo']}_{platformID}_country.csv",
                  index=None, na_rep='')

mismatches? 
_merge
both          130392
left_only          0
right_only         0
Name: count, dtype: int64


# combine views & country

country is a percentage by video. In the next section the actual metric is calculated (video views) per country. This is then summed to the profile level. 
As the average is larger than 1 the country view is rebased 


In [19]:

# post_url: link
ttk_country['w/c'] = pd.to_datetime(ttk_country['w/c'])

ttk_country = ttk_country.drop(columns=['_merge'], errors='ignore')
views_scaled = views_scaled.drop(columns=['_merge'], errors='ignore')

ttk_data = ttk_country.merge(views_scaled, on=['Channel ID', 'link', 'w/c'], how='outer',
                                indicator=True)
print(f"mismatches between datasets! unreviewed yet\n {ttk_data._merge.value_counts()}")
ttk_data = ttk_data[ttk_data['_merge'] == 'both'].drop(columns=['_merge'])

ttk_data['country_views_video_level'] = ttk_data["final_video_views"] * ttk_data["rescaled_percentage"]
ttk_perCountry = ttk_data.groupby(['w/c', 'Channel ID', 'ServiceID',
                                'PlaceID'])['country_views_video_level'].sum().rename('country_views_profile_level').reset_index()
ttk_global = ttk_data.groupby(['w/c', 'ServiceID','Channel ID', 
                              ])['country_views_video_level'].sum().rename('global_views_profile_level').reset_index()

ttk_df = ttk_perCountry.merge(ttk_global, on=['ServiceID', 'w/c', 'Channel ID'], how='outer', 
                           indicator=True)
print(f"no mismatches \n {ttk_df._merge.value_counts()}")
ttk_df = ttk_df.drop(columns=['_merge'])

ttk_df['profile_country_views_%'] = ttk_df['country_views_profile_level']/ttk_df['global_views_profile_level']

ttk_df = ttk_df.merge(views_scaled, on=['ServiceID', 'Channel ID', 'w/c'], how='inner')
ttk_df['uv_by_country'] =  ttk_df['engaged_users'] * ttk_df['profile_country_views_%']

cols = ['w/c', 'PlaceID', 'ServiceID', 'Channel ID', 'uv_by_country', ]
ttk_df[cols].to_csv(f"../data/processed/{platformID}/{gam_info['file_timeinfo']}_{platformID}_uniqueViewer_country.csv", 
                     index=None)


mismatches between datasets! unreviewed yet
 _merge
both          75192
left_only     43120
right_only      161
Name: count, dtype: int64
no mismatches 
 _merge
both          7345
left_only        0
right_only       0
Name: count, dtype: int64
